In [3]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

# =========================================================
# 1) PATH SETUP: Kaggle + Local uyumlu
# =========================================================
KAGGLE_BASE = "/kaggle/input/competitions/playground-series-s6e3/"
LOCAL_BASE = "../data/raw/"

if os.path.exists(os.path.join(KAGGLE_BASE, "train.csv")):
    BASE_PATH = KAGGLE_BASE
    ENVIRONMENT = "Kaggle"
elif os.path.exists(os.path.join(LOCAL_BASE, "train.csv")):
    BASE_PATH = LOCAL_BASE
    ENVIRONMENT = "Local"
else:
    raise FileNotFoundError(
        "train.csv bulunamadı. Kaggle dizinini veya local klasörü kontrol et."
    )

print(f"Çalışma ortamı: {ENVIRONMENT}")
print(f"Kullanılan yol : {BASE_PATH}")

train_path = os.path.join(BASE_PATH, "train.csv")
test_path = os.path.join(BASE_PATH, "test.csv")
sub_path = os.path.join(BASE_PATH, "sample_submission.csv")

# =========================================================
# 2) DATA LOAD
# =========================================================
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sub = pd.read_csv(sub_path)

print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")
print(f"Sub shape  : {sub.shape}")

# =========================================================
# 3) BASIC SETUP
# =========================================================
TARGET = "Churn"
ID_COL = "id"
RANDOM_STATE = 42
N_SPLITS = 3

# Target mapping
train[TARGET] = train[TARGET].map({"No": 0, "Yes": 1})

if train[TARGET].isnull().sum() > 0:
    bad_values = train.loc[train[TARGET].isnull(), TARGET].unique()
    raise ValueError(f"Churn sütununda beklenmeyen değerler var: {bad_values}")

X = train.drop(columns=[ID_COL, TARGET]).copy()
y = train[TARGET].astype(int).copy()
X_test = test.drop(columns=[ID_COL]).copy()

# Kategorik sütunları bul
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

# LightGBM için category tipine çevir
for col in cat_cols:
    X[col] = X[col].astype("category")
    X_test[col] = X_test[col].astype("category")

print(f"\nFeature sayısı    : {X.shape[1]}")
print(f"Kategorik sütun   : {len(cat_cols)}")
print(f"Churn oranı       : {y.mean():.2%}")
print(f"Kategorik kolonlar: {cat_cols}")

# =========================================================
# 4) CROSS VALIDATION
# =========================================================
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="auc",
        categorical_feature=cat_cols
    )

    valid_pred = model.predict_proba(X_valid)[:, 1]
    test_pred = model.predict_proba(X_test)[:, 1]

    oof_preds[valid_idx] = valid_pred
    test_preds += test_pred / N_SPLITS

    fold_auc = roc_auc_score(y_valid, valid_pred)
    fold_scores.append(fold_auc)

    print(f"Fold {fold} AUC: {fold_auc:.5f}")

# =========================================================
# 5) CV RESULT
# =========================================================
cv_auc = roc_auc_score(y, oof_preds)

print("\n" + "=" * 55)
print("LightGBM Baseline CV Results")
print("=" * 55)
print(f"Fold scores : {[round(s, 5) for s in fold_scores]}")
print(f"Mean AUC    : {np.mean(fold_scores):.5f}")
print(f"Std AUC     : {np.std(fold_scores):.5f}")
print(f"OOF AUC     : {cv_auc:.5f}")

# =========================================================
# 6) SUBMISSION
# =========================================================
sub[TARGET] = test_preds

OUTPUT_DIR = "../outputs/submissions"
os.makedirs(OUTPUT_DIR, exist_ok=True)

submission_name = os.path.join(OUTPUT_DIR, "submission_baseline_lightgbm.csv")
sub.to_csv(submission_name, index=False)

print(f"\nSubmission dosyası oluşturuldu: {submission_name}")
print(sub.head())

Çalışma ortamı: Local
Kullanılan yol : ../data/raw/
Train shape: (594194, 21)
Test shape : (254655, 20)
Sub shape  : (254655, 2)

Feature sayısı    : 19
Kategorik sütun   : 15
Churn oranı       : 22.52%
Kategorik kolonlar: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
[LightGBM] [Info] Number of positive: 89211, number of negative: 306918
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009277 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 641
[LightGBM] [Info] Number of data points in the train set: 396129, number of used features: 19
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225207 -> initscore=-1.235576
[LightGBM] [Info] Start training